# Từ bước tiền xử lý bước 2

## Bước này tạo files:

- text_feat.npy (xài sentence_transformer)

## ref:

- https://www.sbert.net/docs/sentence_transformer/pretrained_models.html

In [1]:
import re
import os
import torch
import numpy as np
import pandas as pd

In [4]:
PATH = "../data/2023"

In [5]:
df = pd.read_parquet(os.path.join(PATH, "df_meta.preprocessed.parquet"))

In [6]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 35997 entries, 0 to 35996
Data columns (total 19 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   itemID           35997 non-null  int64  
 1   asin             35997 non-null  str    
 2   main_category    35997 non-null  str    
 3   title            35997 non-null  str    
 4   average_rating   35997 non-null  float64
 5   rating_number    35997 non-null  int64  
 6   features         35997 non-null  object 
 7   description      35997 non-null  object 
 8   price            17404 non-null  float64
 9   images           35997 non-null  object 
 10  videos           35997 non-null  object 
 11  store            35997 non-null  str    
 12  categories       35997 non-null  object 
 13  details          35997 non-null  object 
 14  bought_together  0 non-null      float64
 15  subtitle         0 non-null      float64
 16  author           0 non-null      object 
 17  combined_text    35997 

In [8]:
df.head(3)

,itemID,asin,main_category,title,average_rating,rating_number,features,description,price,images,videos,store,categories,details,bought_together,subtitle,author,combined_text,sentences
0,0,B086QM7FVT,Baby,"Skip Hop Toddler Step Stool, Double Up",4.6,2482,"[A big-kid boost to toddler independence, our ...","[A big-kid boost to toddler independence, our ...",21.99,[{'hi_res': 'https://m.media-amazon.com/images...,"[{'title': 'My favorite step stool', 'url': 'h...",Skip Hop,"[Baby Products, Nursery, Furniture, Storage & ...","{'': None, 'ABPA Partslink Number': None, 'AC ...",NaN,NaN,None,"Skip Hop Toddler Step Stool, Double Up Skip Ho...","Skip Hop Toddler Step Stool, Double Up Skip Ho..."
1,1,B017IQZ9OK,Baby,"Boon Spring Countertop Drying Rack, Green (B11...",4.8,3885,[Drying rack: countertop drying rack holds ite...,[Boon Sprig countertop drying rack easily hold...,12.99,[{'hi_res': 'https://m.media-amazon.com/images...,[{'title': 'Versatile and space saving UPDATE ...,Boon,"[Baby Products, Feeding, Bottle-Feeding, Bottl...","{'': None, 'ABPA Partslink Number': None, 'AC ...",NaN,NaN,None,"Boon Spring Countertop Drying Rack, Green (B11...","Boon Spring Countertop Drying Rack, Green (B11..."
2,2,B08FZJ3YHH,Baby,Toilet Seat Covers Disposable - 20 Pack - Wate...,4.8,7996,[✔️ NO MORE STRESS when you need to use a publ...,[],15.97,[{'hi_res': 'https://m.media-amazon.com/images...,[{'title': 'Toilet Seat Covers Disposable: Fun...,Relyo,"[Baby Products, Potty Training, Seat Covers]","{'': None, 'ABPA Partslink Number': None, 'AC ...",NaN,NaN,None,Toilet Seat Covers Disposable - 20 Pack - Wate...,Toilet Seat Covers Disposable - 20 Pack - Wate...


## Text Feature Extraction

In [ ]:
# %pip install sentence_transformers

Note: you may need to restart the kernel to use updated packages.


In [ ]:
# %pip install ipywidgets

In [4]:
# from fastembed import TextEmbedding

# # https://qdrant.github.io/fastembed/examples/Supported_Models/#supported-text-embedding-models
# model = TextEmbedding(model_name="BAAI/bge-small-en-v1.5")
# print("Đang encode...")

# embeddings_generator = model.embed(sentences)
# sentence_embeddings = np.array(list(embeddings_generator))

# print(f"Shape: {sentence_embeddings.shape}")


# np.save(os.path.join('text_feat.npy'), sentence_embeddings)
# print('Đã lưu xong bằng FastEmbed!')

In [245]:
!nvidia-smi

Sat Mar 28 22:31:52 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 591.86                 Driver Version: 591.86         CUDA Version: 13.1     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                  Driver-Model | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 4050 ...  WDDM  |   00000000:01:00.0  On |                  N/A |
| N/A   39C    P4             11W /   74W |    1368MiB /   6141MiB |     31%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [59]:
sentences = df["sentences"].tolist()

In [60]:
sentences[:3]

['Skip Hop Toddler Step Stool, Double Up Skip Hop Baby Baby Products Nursery Furniture Storage  &  Organization Step Stools A big-kid boost to toddler independence, our 2-in-1 step stool makes everything easier for kids to reach. Perfect at the bathroom sink and toilet for potty training, it’s also great as a kitchen helper and more Use the space-saving nesting stools together as a stepped design for an easier climb, or separately Interlocking design keeps stools securely attached when used together Featuring: Wide standing platforms, Non-slip bases  &  treads, Space-saving nesting design Size (inches): Large step stool: 13.25W x 8.5H x 10.25D; Small step stool: 10.375W x 6H x 10.25D A big-kid boost to toddler independence, our 2-in-1 step stool makes everything easier for kids to reach. Perfect at the bathroom sink and toilet for potty training, it’s also great as a kitchen helper and more. Use the space-saving nesting stools together as a stepped design for an easier climb, or separa

In [61]:
print("Bắt đầu Encode... Vui lòng đợi trong giây lát.")
with torch.no_grad():
    sentence_embeddings = model.encode(
        sentences, batch_size=64, show_progress_bar=True, convert_to_numpy=True
    )

print(f"Text encoded! Shape: {sentence_embeddings.shape}")

assert sentence_embeddings.shape[0] == df.shape[0]
save_path = os.path.join(PATH, "text_feat.npy")
np.save(save_path, sentence_embeddings)

print(f"Done! File đã được lưu tại: {save_path}")

Bắt đầu Encode... Vui lòng đợi trong giây lát.


Batches: 100%|██████████| 563/563 [12:35<00:00,  1.34s/it]


Text encoded! Shape: (35997, 384)
Done! File đã được lưu tại: ./data/2023\text_feat.npy


In [ ]:
# !pip install nvitop

In [ ]:
# !nvitop

In [62]:
sentence_embeddings[:10]

array([[-0.02511088,  0.050413  ,  0.04400188, ...,  0.04795245,
         0.07556733,  0.05827542],
       [-0.02860759,  0.0548236 , -0.00681664, ..., -0.03452795,
        -0.04217557,  0.05453426],
       [ 0.0636656 ,  0.01261443,  0.00849598, ...,  0.02520804,
         0.04014093,  0.06704497],
       ...,
       [ 0.02174732,  0.01410911, -0.00569856, ...,  0.05042551,
        -0.0428423 , -0.02519943],
       [ 0.03739659, -0.02329124,  0.00790992, ...,  0.02438011,
         0.04282311, -0.04832435],
       [-0.04577953, -0.00894414,  0.06457003, ...,  0.03238399,
        -0.00303201,  0.02178374]], shape=(10, 384), dtype=float32)

## Image Feature Extraction


In [ ]:
df = pd.read_parquet(os.path.join(PATH, "df_meta.2feat-encoder.parquet"))

In [ ]:
df[:5]

,itemID,asin,categories,description,title,price,imUrl,brand,related,salesRank,combined_text,sentences
0,0,097293751X,[[Baby]],Easily keep track of your baby's or child's da...,"Baby Tracker&reg; - Daily Childcare Journal, S...",17.00,http://ecx.images-amazon.com/images/I/41Bb6wf%...,Time Too,"{'also_bought': ['9729375011', 'B004FN1AE8', '...",None,"Baby Tracker&reg; - Daily Childcare Journal, S...","Baby Tracker&reg; - Daily Childcare Journal, S..."
1,1,9729375011,[[Baby]],This is version of the award-winningBaby Track...,Newborn Baby Tracker&reg; - Round the Clock Ch...,15.95,http://ecx.images-amazon.com/images/I/51r3BLpL...,,"{'also_bought': ['B000V5KPZ4', 'B001F8TLLU', '...",None,Newborn Baby Tracker&reg; - Round the Clock Ch...,Newborn Baby Tracker&reg; - Round the Clock Ch...
2,2,B00000IZQI,[[Baby]],This colorful car collection develops motor sk...,Fisher Price Nesting Action Vehicles,8.37,http://ecx.images-amazon.com/images/I/51E83QCC...,,"{'also_bought': ['B0042D69W4', 'B00428LIZM', '...",None,Fisher Price Nesting Action Vehicles This colo...,Fisher Price Nesting Action Vehicles This colo...
3,3,B00000J3LL,[[Baby]],This darling cloth book offers hands-on experi...,"My Quiet Book, Fabric Activity Book for Children",27.00,http://ecx.images-amazon.com/images/I/51GoNXhB...,,"{'also_bought': ['B00000J3LC', 'B0043G4JOA', '...",None,"My Quiet Book, Fabric Activity Book for Childr...","My Quiet Book, Fabric Activity Book for Childr..."
4,4,B00002JV9S,[[Baby]],"In a relatively new concept in teething, The F...",The First Years Massaging Action Teether,8.84,http://ecx.images-amazon.com/images/I/41gVp98n...,The First Years,"{'also_bought': ['B0013FCBJO', 'B0019QCGVK', '...",None,The First Years Massaging Action Teether The F...,The First Years Massaging Action Teether The F...


In [6]:
import array


def read_image_features(path, feature_size):
    if not os.path.exists(path):
        return
    with open(path, "rb") as f:
        while True:
            asin_bytes = f.read(10)
            if not asin_bytes:
                break
            try:
                asin = asin_bytes.decode("utf-8").strip()
                a = array.array("f")
                # Đọc đúng số lượng float bạn yêu cầu
                a.fromfile(f, feature_size)
                yield asin, a.tolist()
            except EOFError:
                break
            except Exception as e:
                print(f"Lỗi tại vị trí {f.tell()}: {e}")
                break

In [7]:
# --- CẤU HÌNH ---
FEATURE_SIZE = 4096
FILE_B_PATH = os.path.join(PATH, "image_feature.vgg16.b")

In [8]:
image_features = read_image_features(FILE_B_PATH, feature_size=FEATURE_SIZE)

In [44]:
# Lấy thử 1 phần tử đầu tiên
first_item = next(image_features)

# Kiểm tra hình dạng
print(f"Kiểu dữ liệu của 1 phần tử: {type(first_item)}")
print(f"Mã ASIN (ID sản phẩm): {first_item[0]}")
print(f"Độ dài vector ảnh: {len(first_item[1])}")
print(f"5 giá trị đầu tiên trong vector: {first_item[1][:5]}")

Kiểu dữ liệu của 1 phần tử: <class 'tuple'>
Mã ASIN (ID sản phẩm): B004LE8ZYO
Độ dài vector ảnh: 4096
5 giá trị đầu tiên trong vector: [0.0, 0.0, 0.0, 0.0, 0.0]


In [45]:
first_item[1][:10]

[0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.6348890066146851,
 0.0,
 0.10182002186775208,
 0.0,
 0.0]

In [ ]:
from tqdm import tqdm

num_items = len(df)
OUTPUT_NPY = os.path.join(PATH, "image_feat.npy")

# 1. Map ASIN -> itemID
map_asin_itemID = dict(zip(df["asin"], df["itemID"]))

# 2. Khởi tạo ma trận và biến hỗ trợ tính trung bình
final_matrix = np.zeros((num_items, FEATURE_SIZE), dtype=np.float32)
filled_indices = set()
running_sum = np.zeros(
    FEATURE_SIZE, dtype=np.float64
)  # Dùng float64 để tránh tràn số khi cộng dồn

print(f"🚀 Đang trích xuất ảnh cho {num_items} sản phẩm...")

# 3. VỪA ĐỌC VỪA ĐIỀN + TÍNH TỔNG CỘNG DỒN
for asin, feat_list in tqdm(
    read_image_features(FILE_B_PATH, FEATURE_SIZE), desc="Processing"
):
    if asin in map_asin_itemID:
        target_idx = int(map_asin_itemID[asin])
        feat_array = np.array(feat_list, dtype=np.float32)

        final_matrix[target_idx] = feat_array
        filled_indices.add(target_idx)

        # Cộng dồn để lát nữa tính trung bình (chỉ tính trên những cái có dữ liệu thật)
        running_sum += feat_array

# 4. XỬ LÝ DỮ LIỆU THIẾU (Lấp đầy bằng Average Vector)
all_indices = set(range(num_items))
missing_indices = sorted(list(all_indices - filled_indices))

if len(filled_indices) > 0:
    # Tính vector trung bình từ những item CÓ ảnh
    avg_vector = (running_sum / len(filled_indices)).astype(np.float32)

    if missing_indices:
        print(
            f"⚠️ Cảnh báo: Thiếu {len(missing_indices)} ảnh. Đang điền bằng vector trung bình..."
        )
        # Điền vector trung bình vào các vị trí trống bằng vectorization (nhanh hơn loop)
        final_matrix[missing_indices] = avg_vector

        # Lưu log danh sách ID thiếu để bạn kiểm tra đồ án
        err_log = os.path.join(PATH, "missed_img_itemIDs.csv")
        np.savetxt(err_log, missing_indices, delimiter=",", fmt="%d")
else:
    print("❌ LỖI NGHIÊM TRỌNG: Không tìm thấy bất kỳ ảnh nào khớp với dữ liệu!")
    # Tùy chọn: điền toàn bộ bằng 0 hoặc raise lỗi nếu cần

# 5. LƯU KẾT QUẢ
np.save(OUTPUT_NPY, final_matrix)

print(f"✅ Hoàn thành! File đã lưu: {OUTPUT_NPY}")
print(f"Kích thước ma trận: {final_matrix.shape}")
print(f"Số lượng item dùng ảnh mặc định: {len(missing_indices)}")

🚀 Đang trích xuất ảnh cho 7050 sản phẩm...


Processing: 7037it [00:00, 7160.80it/s]


⚠️ Cảnh báo: Thiếu 13 ảnh. Đang điền bằng vector trung bình...
✅ Hoàn thành! File đã lưu: ./data/2014\image_feat.npy
Kích thước ma trận: (7050, 4096)
Số lượng item dùng ảnh mặc định: 13


In [ ]:
# 1. Giai đoạn 1: Lọc và Thu thập dữ liệu hiện có
item2id = dict(zip(df["asin"], df["itemID"]))
img_data = read_image_features(FILE_B_PATH, feature_size=FEATURE_SIZE)

feats = {}
avg = []
for d in img_data:
    if d[0] in item2id:
        feats[int(item2id[d[0]])] = d[1]
        avg.append(d[1])

# 2. Giai đoạn 2: Tính toán "Vector trung bình" (Imputation)
avg = np.array(avg).mean(0).tolist()

In [53]:
# 3. Giai đoạn 3: Lấp đầy khoảng trống (Final Mapping)
ret = []
non_no = []
for i in range(len(item2id)):
    if i in feats:
        ret.append(feats[i])  # Nếu có ảnh: lấy ảnh thật
    else:
        non_no.append(i)  # Nếu thiếu ảnh: lưu lại ID bị thiếu
        ret.append(avg)  # ...và gán cho nó vector trung bình

print("# of items not in processed image features:", len(non_no))
assert len(ret) == len(item2id)
np.save("image_feat.npy", np.array(ret))
np.savetxt(
    "missed_img_itemIDs.csv", non_no, delimiter=",", fmt="%d"
)  # save for report missing data
print("done!")

# of items not in processed image features: 13
done!
